# 1. Memuat Dataset

Tahap pertama dalam proses analisis adalah memuat dataset menggunakan library **Pandas**. Dataset `data.zip` dibaca menggunakan fungsi `read_csv()` dengan pemisah data berupa titik koma (`;`).

Setelah dataset berhasil dimuat, dilakukan beberapa pemeriksaan awal, yaitu:

- Menampilkan 5 baris pertama menggunakan `head()` untuk melihat contoh isi data.
- Menampilkan ukuran dataset menggunakan `shape` untuk mengetahui jumlah baris dan kolom.
- Menampilkan informasi struktur dataset menggunakan `info()` untuk mengetahui nama kolom, jumlah data yang terisi, serta tipe data pada setiap kolom.

Pemeriksaan awal ini dilakukan untuk memahami kondisi dataset sebelum masuk ke tahap *data cleaning* dan analisis lebih lanjut.

In [4]:
import pandas as pd
import numpy as np

# Membaca dataset
df = pd.read_csv("../data/raw/indonesia_ecommerce_sales_challenging.csv", sep=";")

# Melihat 5 baris pertama
display(df.head())

# Melihat jumlah baris dan kolom
print("Ukuran dataset:", df.shape)

# Melihat struktur dataset
df.info()

,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,order_date
0,ORD_0000001,2.0,2000,0,0,Celengan,1,Selesai,Tidak dibatalkan,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,2024-04-01
1,ORD_0000002,1.0,500,0,0,Celengan,1,Selesai,Tidak dibatalkan,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,2024-04-01
2,ORD_0000003,1.0,500,0,0,Celengan,1,Selesai,Tidak dibatalkan,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,2024-04-01
3,ORD_0000004,2.0,400,0,0,Mangkok Sambal / Saus,1,Selesai,Tidak dibatalkan,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,2024-04-01
4,ORD_0000005,3.0,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,2024-04-01


Ukuran dataset: (18868, 19)
<class 'pandas.DataFrame'>
RangeIndex: 18868 entries, 0 to 18867
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   order_id                            18868 non-null  str    
 1   total_qty                           18585 non-null  float64
 2   total_weight_gr                     18868 non-null  int64  
 3   total_returned_qty                  18868 non-null  int64  
 4   Total Diskon                        18868 non-null  int64  
 5   product_categories                  18868 non-null  str    
 6   num_product_categories              18868 non-null  int64  
 7   Status Pesanan                      18868 non-null  str    
 8   Alasan Pembatalan                   18868 non-null  str    
 9   Opsi Pengiriman                     18868 non-null  str    
 10  Metode Pembayaran                   18585 non-null  str    
 11  Kota/Kabupaten          

# 2. Menentukan Kolom Relevan untuk Analisis

Sebelum melakukan *data cleaning*, kolom yang digunakan dalam analisis ditentukan terlebih dahulu berdasarkan pertanyaan bisnis.

Terdapat tiga pertanyaan utama yang akan dianalisis:

1. **Produk atau kategori apa yang menghasilkan pendapatan tertinggi?**
   - `product_categories`
   - `total_pembayaran`
   - `status_pesanan`

2. **Bagaimana tren penjualan setiap tahun?**
   - `order_date`
   - `total_pembayaran`
   - `status_pesanan`

3. **Kategori apa yang paling banyak dibeli di Kabupaten Bogor?**
   - `product_categories`
   - `total_qty`
   - `kota_kabupaten`
   - `status_pesanan`

Kolom `status_pesanan` digunakan untuk membedakan transaksi yang berhasil dengan transaksi yang dibatalkan.

Khusus untuk `product_categories`, perlu diperhatikan bahwa satu transaksi dapat berisi lebih dari satu kategori. Oleh karena itu, pada analisis kategori perlu dilakukan pemeriksaan dan pemisahan kategori agar hasil analisis tidak keliru.

In [2]:
# Menampilkan kolom yang tersedia dalam dataset
print("Daftar kolom:")
print(df.columns.tolist())

# Menampilkan beberapa kolom yang relevan
display(
    df[
        [
            "product_categories",
            "Total Pembayaran",
            "Status Pesanan",
            "total_qty",
            "Kota/Kabupaten",
            "order_date"
        ]
    ].head()
)

Daftar kolom:
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Waktu Pesanan Dibuat', 'order_date']


,product_categories,Total Pembayaran,Status Pesanan,total_qty,Kota/Kabupaten,order_date
0,Celengan,38300,Selesai,2.0,KOTA SERANG,2024-04-01
1,Celengan,18576,Selesai,1.0,KOTA SEMARANG,2024-04-01
2,Celengan,7069,Selesai,1.0,KAB. BOGOR,2024-04-01
3,Mangkok Sambal / Saus,32200,Selesai,2.0,KOTA JAMBI,2024-04-01
4,"Keranjang, Other, Tempat Nasi",0,Batal,3.0,KOTA TANGERANG,2024-04-01


# 3. Pemeriksaan Kondisi dan Karakteristik Dataset

Setelah dataset berhasil dimuat, dilakukan pemeriksaan lebih lanjut untuk mengetahui kondisi dan karakteristik data.

Pemeriksaan meliputi:

- **Missing values**, untuk mengetahui jumlah data yang kosong pada setiap kolom.
- **Data duplikat**, untuk memastikan tidak terdapat transaksi yang tercatat lebih dari satu kali.
- **Status pesanan**, untuk mengetahui kondisi transaksi seperti `Selesai`, `Batal`, dan status lainnya.
- **Kategori produk**, untuk mengetahui kategori yang paling sering muncul.
- **Wilayah**, untuk mengetahui wilayah yang terdapat dalam dataset.
- **Rentang tanggal**, untuk mengetahui periode transaksi yang tersedia.
- **Statistik deskriptif**, untuk mengetahui gambaran umum data numerik seperti jumlah barang, pembayaran, dan diskon.

Hasil pemeriksaan ini digunakan sebagai dasar untuk menentukan proses *data cleaning* berikutnya.

In [3]:
# 1. Memeriksa missing values
print("Missing values:")
display(
    df.isnull()
    .sum()
    .to_frame("Jumlah Missing")
)

# 2. Memeriksa data duplikat
print("Jumlah baris duplikat:", df.duplicated().sum())

# 3. Memeriksa status pesanan
print("\nStatus Pesanan:")
display(
    df["Status Pesanan"]
    .value_counts(dropna=False)
)

# 4. Memeriksa kategori produk
print("\n10 kategori produk yang paling sering muncul:")
display(
    df["product_categories"]
    .value_counts()
    .head(10)
)

# 5. Memeriksa wilayah
print("\n10 wilayah yang paling sering muncul:")
display(
    df["Kota/Kabupaten"]
    .value_counts()
    .head(10)
)

# 6. Memeriksa rentang tanggal
print("\nTanggal awal:", df["order_date"].min())
print("Tanggal akhir:", df["order_date"].max())

# 7. Statistik deskriptif
display(
    df[
        [
            "total_qty",
            "Total Pembayaran",
            "Total Diskon"
        ]
    ].describe()
)

Missing values:


,Jumlah Missing
order_id,0
total_qty,283
total_weight_gr,0
total_returned_qty,0
Total Diskon,0
product_categories,0
num_product_categories,0
Status Pesanan,0
Alasan Pembatalan,0
Opsi Pengiriman,0


Jumlah baris duplikat: 0

Status Pesanan:


Status Pesanan
Selesai                                                                                   16045
Batal                                                                                      2573
Sedang Dikirim                                                                               59
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.       38
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.       34
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.       34
Telah Dikirim                                                                                30
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-05.       25
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-09.       22
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-10.        6
Pesanan diterima, namun P


10 kategori produk yang paling sering muncul:


product_categories
Celengan                  5083
Mangkok Sambal / Saus     3253
Aksesoris Pintu           2365
Nampan / Tray             1140
Seal / Baut / Roof         586
Rak / Rak Serbaguna        573
Baskom / Mangkok Besar     536
Lunch Box / Rantang        492
Other                      473
Keranjang                  457
Name: count, dtype: int64


10 wilayah yang paling sering muncul:


Kota/Kabupaten
KAB. BOGOR                1082
KAB. TANGERANG            1027
KOTA TANGERANG             915
KAB. BEKASI                750
KOTA TANGERANG SELATAN     740
KOTA JAKARTA BARAT         681
KOTA JAKARTA SELATAN       634
KOTA JAKARTA TIMUR         616
KOTA BEKASI                564
KOTA DEPOK                 541
Name: count, dtype: int64


Tanggal awal: 01/02/2025
Tanggal akhir: 31/08/2025


,total_qty,Total Pembayaran,Total Diskon
count,18585.000000,1.886800e+04,18868.000000
mean,2.565940,5.099066e+04,424.220691
std,7.799706,1.426189e+05,10110.492750
min,1.000000,0.000000e+00,0.000000
25%,1.000000,1.367500e+04,0.000000
50%,1.000000,2.180000e+04,0.000000
75%,2.000000,4.231125e+04,0.000000
max,256.000000,3.076000e+06,700000.000000
